<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day15_(260423)_API_Routes_%EA%B0%9C%EB%B0%9C_%EB%B0%8F_Middleware%EB%A5%BC_%ED%99%9C%EC%9A%A9%ED%95%9C_%EC%9D%B8%EC%A6%9D_%EC%9D%B8%EA%B0%80_%EC%B2%98%EB%A6%AC_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile jwt-httponly-lab/scripts/schema.sql

CREATE TABLE users (
  id INT AUTO_INCREMENT PRIMARY KEY,
  name VARCHAR(100) NOT NULL,
  email VARCHAR(191) NOT NULL UNIQUE,
  password_hash VARCHAR(255) NOT NULL,
  role ENUM('user', 'admin') NOT NULL DEFAULT 'user',
  status ENUM('active', 'inactive') NOT NULL DEFAULT 'active',
  created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
  updated_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP
);

Writing jwt-httponly-lab/scripts/schema.sql


In [2]:
%%writefile jwt-httponly-lab/scripts/seed-admin.ts

import bcrypt from 'bcryptjs';
import mysql from 'mysql2/promise';

async function seedAdmin() {
  const connection = await mysql.createConnection({
    host: process.env.DB_HOST || '127.0.0.1',
    port: Number(process.env.DB_PORT || 3306),
    user: process.env.DB_USER || 'testuser',
    password: process.env.DB_PASSWORD || '1234',
    database: process.env.DB_NAME || 'jwt_lab'
  });

  const passwordHash = await bcrypt.hash('1234', 10);

  await connection.execute(
    `
    INSERT INTO users (name, email, password_hash, role, status)
    VALUES (?, ?, ?, ?, ?)
    ON DUPLICATE KEY UPDATE
      name = VALUES(name),
      password_hash = VALUES(password_hash),
      role = VALUES(role),
      status = VALUES(status)
    `,
    ['관리자', 'admin@test.com', passwordHash, 'admin', 'active']
  );

  await connection.end();
  console.log('관리자 계정 시드 완료');
}

seedAdmin().catch((error) => {
  console.error(error);
  process.exit(1);
});

Writing jwt-httponly-lab/scripts/seed-admin.ts


In [3]:
%%writefile jwt-httponly-lab/lib/db.ts

import mysql from 'mysql2/promise';

const pool = mysql.createPool({
  host: process.env.DB_HOST || '127.0.0.1',
  port: Number(process.env.DB_PORT || 3306),
  user: process.env.DB_USER || 'testuser',
  password: process.env.DB_PASSWORD || '1234',
  database: process.env.DB_NAME || 'jwt_lab',
  waitForConnections: true,
  connectionLimit: 10,
  queueLimit: 0
});

export default pool;

Writing jwt-httponly-lab/lib/db.ts


In [4]:
%%writefile jwt-httponly-lab/lib/auth.ts

import bcrypt from 'bcryptjs';
import { SignJWT, jwtVerify } from 'jose';

export type JwtPayload = {
  userId: number;
  email: string;
  role: 'user' | 'admin';
};

const secretKey = new TextEncoder().encode(
  process.env.JWT_SECRET || 'replace_this_with_a_long_random_secret'
);

export async function hashPassword(password: string) {
  return bcrypt.hash(password, 10);
}

export async function comparePassword(
  password: string,
  passwordHash: string
) {
  return bcrypt.compare(password, passwordHash);
}

export async function signToken(payload: JwtPayload) {
  return new SignJWT(payload)
    .setProtectedHeader({ alg: 'HS256' })
    .setIssuedAt()
    .setExpirationTime('1h')
    .sign(secretKey);
}

export async function verifyToken(token: string) {
  const { payload } = await jwtVerify(token, secretKey);

  return {
    userId: Number(payload.userId),
    email: String(payload.email),
    role: payload.role as 'user' | 'admin'
  };
}

Writing jwt-httponly-lab/lib/auth.ts


In [5]:
%%writefile jwt-httponly-lab/lib/session.ts

import { cookies } from 'next/headers';
import { verifyToken } from '@/lib/auth';

export async function getSessionUser() {
  const cookieStore = await cookies();
  const cookieName = process.env.TOKEN_COOKIE_NAME || 'token';
  const token = cookieStore.get(cookieName)?.value;

  if (!token) {
    return null;
  }

  try {
    const user = await verifyToken(token);
    return user;
  } catch {
    return null;
  }
}

Writing jwt-httponly-lab/lib/session.ts


In [6]:
%%writefile jwt-httponly-lab/lib/validators.ts

export function validateSignupInput(data: {
  name?: string;
  email?: string;
  password?: string;
}) {
  const name = data.name?.trim() || '';
  const email = data.email?.trim() || '';
  const password = data.password?.trim() || '';

  if (!name) {
    return '이름을 입력해야 합니다.';
  }

  if (!email) {
    return '이메일을 입력해야 합니다.';
  }

  if (!password) {
    return '비밀번호를 입력해야 합니다.';
  }

  if (password.length < 4) {
    return '비밀번호는 최소 4자 이상이어야 합니다.';
  }

  return null;
}

export function validateLoginInput(data: {
  email?: string;
  password?: string;
}) {
  const email = data.email?.trim() || '';
  const password = data.password?.trim() || '';

  if (!email) {
    return '이메일을 입력해야 합니다.';
  }

  if (!password) {
    return '비밀번호를 입력해야 합니다.';
  }

  return null;
}

Writing jwt-httponly-lab/lib/validators.ts


In [7]:
%%writefile jwt-httponly-lab/lib/responses.ts

import { NextResponse } from 'next/server';

export function badRequest(message: string) {
  return NextResponse.json({ message }, { status: 400 });
}

export function unauthorized(message = '인증이 필요합니다.') {
  return NextResponse.json({ message }, { status: 401 });
}

export function forbidden(message = '권한이 없습니다.') {
  return NextResponse.json({ message }, { status: 403 });
}

export function ok(data: unknown) {
  return NextResponse.json(data, { status: 200 });
}

export function created(data: unknown) {
  return NextResponse.json(data, { status: 201 });
}

Writing jwt-httponly-lab/lib/responses.ts


In [8]:
%%writefile jwt-httponly-lab/app/api/auth/signup/route.ts

import { RowDataPacket } from 'mysql2';
import pool from '@/lib/db';
import { hashPassword } from '@/lib/auth';
import { created, badRequest } from '@/lib/responses';
import { validateSignupInput } from '@/lib/validators';

type UserRow = RowDataPacket & {
  id: number;
};

export async function POST(request: Request) {
  const body = await request.json();
  const validationError = validateSignupInput(body);

  if (validationError) {
    return badRequest(validationError);
  }

  const name = body.name.trim();
  const email = body.email.trim();
  const password = body.password.trim();

  const [existingUsers] = await pool.query<UserRow[]>(
    `
    SELECT id
    FROM users
    WHERE email = ?
    LIMIT 1
    `,
    [email]
  );

  if (existingUsers.length > 0) {
    return badRequest('이미 사용 중인 이메일입니다.');
  }

  const passwordHash = await hashPassword(password);

  await pool.execute(
    `
    INSERT INTO users (name, email, password_hash, role, status)
    VALUES (?, ?, ?, 'user', 'active')
    `,
    [name, email, passwordHash]
  );

  return created({
    message: '회원가입이 완료되었습니다.'
  });
}

Writing jwt-httponly-lab/app/api/auth/signup/route.ts


In [9]:
%%writefile jwt-httponly-lab/app/api/auth/login/route.ts

import { NextResponse } from 'next/server';
import { RowDataPacket } from 'mysql2';
import pool from '@/lib/db';
import { comparePassword, signToken } from '@/lib/auth';
import { badRequest, unauthorized } from '@/lib/responses';
import { validateLoginInput } from '@/lib/validators';

type LoginUserRow = RowDataPacket & {
  id: number;
  name: string;
  email: string;
  password_hash: string;
  role: 'user' | 'admin';
  status: 'active' | 'inactive';
};

export async function POST(request: Request) {
  const body = await request.json();
  const validationError = validateLoginInput(body);

  if (validationError) {
    return badRequest(validationError);
  }

  const email = body.email.trim();
  const password = body.password.trim();

  const [users] = await pool.query<LoginUserRow[]>(
    `
    SELECT id, name, email, password_hash, role, status
    FROM users
    WHERE email = ?
    LIMIT 1
    `,
    [email]
  );

  if (users.length === 0) {
    return unauthorized('이메일 또는 비밀번호가 올바르지 않습니다.');
  }

  const user = users[0];

  if (user.status !== 'active') {
    return unauthorized('비활성화된 계정입니다.');
  }

  const isMatched = await comparePassword(password, user.password_hash);

  if (!isMatched) {
    return unauthorized('이메일 또는 비밀번호가 올바르지 않습니다.');
  }

  const token = await signToken({
    userId: user.id,
    email: user.email,
    role: user.role
  });

  const response = NextResponse.json(
    {
      message: '로그인 성공'
    },
    { status: 200 }
  );

  const cookieName = process.env.TOKEN_COOKIE_NAME || 'token';

  response.cookies.set(cookieName, token, {
    httpOnly: true,
    secure: process.env.NODE_ENV === 'production',
    sameSite: 'lax',
    path: '/',
    maxAge: 60 * 60
  });

  return response;
}

Writing jwt-httponly-lab/app/api/auth/login/route.ts


In [10]:
%%writefile jwt-httponly-lab/app/api/auth/logout/route.ts

import { NextResponse } from 'next/server';

export async function POST() {
  const response = NextResponse.json(
    {
      message: '로그아웃 완료'
    },
    { status: 200 }
  );

  const cookieName = process.env.TOKEN_COOKIE_NAME || 'token';

  response.cookies.set(cookieName, '', {
    httpOnly: true,
    secure: process.env.NODE_ENV === 'production',
    sameSite: 'lax',
    path: '/',
    maxAge: 0
  });

  return response;
}

Writing jwt-httponly-lab/app/api/auth/logout/route.ts


In [11]:
%%writefile jwt-httponly-lab/app/api/auth/me/route.ts

import { ok, unauthorized } from '@/lib/responses';
import { getSessionUser } from '@/lib/session';

export async function GET() {
  const user = await getSessionUser();

  if (!user) {
    return unauthorized('인증이 필요합니다.');
  }

  return ok({
    user
  });
}

Writing jwt-httponly-lab/app/api/auth/me/route.ts


In [12]:
%%writefile jwt-httponly-lab/app/api/admin/users/route.ts

import { RowDataPacket } from 'mysql2';
import pool from '@/lib/db';
import { forbidden, ok, unauthorized } from '@/lib/responses';
import { getSessionUser } from '@/lib/session';

type UserListRow = RowDataPacket & {
  id: number;
  name: string;
  email: string;
  role: 'user' | 'admin';
  status: 'active' | 'inactive';
  created_at: string;
};

export async function GET() {
  const user = await getSessionUser();

  if (!user) {
    return unauthorized('인증이 필요합니다.');
  }

  if (user.role !== 'admin') {
    return forbidden('관리자 권한이 필요합니다.');
  }

  const [rows] = await pool.query<UserListRow[]>(
    `
    SELECT id, name, email, role, status, created_at
    FROM users
    ORDER BY id ASC
    `
  );

  return ok({
    users: rows
  });
}

Writing jwt-httponly-lab/app/api/admin/users/route.ts


In [13]:
%%writefile jwt-httponly-lab/middleware.ts

import { NextRequest, NextResponse } from 'next/server';
import { verifyToken } from '@/lib/auth';

const protectedPaths = ['/practice/user', '/practice/admin'];
const adminPaths = ['/practice/admin'];

export async function middleware(request: NextRequest) {
  const cookieName = process.env.TOKEN_COOKIE_NAME || 'token';
  const token = request.cookies.get(cookieName)?.value;
  const { pathname } = request.nextUrl;

  const isProtectedPath = protectedPaths.some((path) =>
    pathname.startsWith(path)
  );

  const isAdminPath = adminPaths.some((path) =>
    pathname.startsWith(path)
  );

  if (isProtectedPath && !token) {
    return NextResponse.redirect(new URL('/', request.url));
  }

  if (token) {
    try {
      const user = await verifyToken(token);

      if (isAdminPath && user.role !== 'admin') {
        return NextResponse.redirect(new URL('/practice/user', request.url));
      }
    } catch {
      return NextResponse.redirect(new URL('/', request.url));
    }
  }

  return NextResponse.next();
}

export const config = {
  matcher: ['/practice/user/:path*', '/practice/admin/:path*']
};

Writing jwt-httponly-lab/middleware.ts


In [14]:
%%writefile jwt-httponly-lab/app/layout.tsx

export default function RootLayout({
  children,
}: Readonly<{
  children: React.ReactNode;
}>) {
  return (
    <html lang="ko">
      <body>{children}</body>
    </html>
  );
}

Writing jwt-httponly-lab/app/layout.tsx


In [15]:
%%writefile jwt-httponly-lab/app/page.tsx

export default function HomePage() {
  return (
    <main style={{ padding: '24px', fontFamily: 'Arial, sans-serif' }}>
      <h1>JWT + HttpOnly 쿠키 인증 실습</h1>
      <p>이 프로젝트는 UI보다 API 구조 이해를 목표로 합니다.</p>

      <h2>테스트 대상 API</h2>
      <ul>
        <li>POST /api/auth/signup</li>
        <li>POST /api/auth/login</li>
        <li>POST /api/auth/logout</li>
        <li>GET /api/auth/me</li>
        <li>GET /api/admin/users</li>
      </ul>

      <h2>연습용 페이지</h2>
      <ul>
        <li>/practice/user</li>
        <li>/practice/admin</li>
      </ul>
    </main>
  );
}

Writing jwt-httponly-lab/app/page.tsx


In [16]:
%%writefile jwt-httponly-lab/app/practice/user/page.tsx

export default function PracticeUserPage() {
  return (
    <main style={{ padding: '24px', fontFamily: 'Arial, sans-serif' }}>
      <h1>사용자 전용 연습 페이지</h1>
      <p>로그인한 사용자만 접근할 수 있습니다.</p>
      <p>관리자도 접근 가능합니다.</p>
    </main>
  );
}

Writing jwt-httponly-lab/app/practice/user/page.tsx


In [17]:
%%writefile jwt-httponly-lab/app/practice/admin/page.tsx

export default function PracticeAdminPage() {
  return (
    <main style={{ padding: '24px', fontFamily: 'Arial, sans-serif' }}>
      <h1>관리자 전용 연습 페이지</h1>
      <p>관리자 role을 가진 사용자만 접근할 수 있습니다.</p>
    </main>
  );
}

Writing jwt-httponly-lab/app/practice/admin/page.tsx
